# 18 · The sentence path of the compiled monad (paper §8)
Code: `VAPMIP/PtolC/ptol.c` (`console_speak`, `infer_speech_act`, `get_monad_sentence`), `VAPMIP/engine/grammar/ptolc_bridge.py`.
`ptol -say` runs with the **real** HOME (it writes no files; the bridge needs the user's site-packages and `~/nltk_data`).

In [1]:
from pn import *
L = Ledger("18 · the sentence path of the compiled monad")
rows = []
for p in ["what is a tree", "finger pointer", "can you play the piano?"]:
    s, status, dt = ptol_say(p); rows.append((p, s, status, dt)); print(f"{p!r:30} -> {s!r}   {status}   {dt:.1f} s")
L.check("ptol -say returns one non-empty line for each prompt", all(r[1] for r in rows))
L.check("the status line reports σ_self, Γ and the firing primes", all("σ=" in r[2] and "Γ=" in r[2] for r in rows))

'what is a tree'               -> 'Time is able.'   [σ=0.2100 Γ=-0.5800 primes: 37 41 43 47 53]   13.8 s


'finger pointer'               -> 'Wittgenstein trolleys wittgenstein.'   [σ=0.1501 Γ=-0.6998 primes: 37 41 43 47 53]   12.9 s


'can you play the piano?'      -> 'Time is able.'   [σ=0.0307 Γ=-0.9385 primes: 41 43 47 53]   12.5 s


True

## The bridge on its own: three lines in, one line out

In [2]:
def bridge(text):
    r = subprocess.run(["python3", "-m", "engine.grammar.ptolc_bridge"], input=text, capture_output=True, text=True, cwd=VAPMIP, timeout=120)
    return r.returncode, r.stdout.strip()
for txt in ("GAMMA -0.58\nACT define_X\nWORDS tree\n", "GAMMA 0.30\nACT how\nWORDS tree plant grow\n", "GAMMA -0.58\nACT define_X\nWORDS\n"):
    print(repr(txt), "->", bridge(txt))
rc, out = bridge("GAMMA -0.58\nACT define_X\nWORDS tree\n")
L.check("a valid word bag gives exit 0 and one sentence", rc == 0 and out != "", out)
rc, out = bridge("GAMMA -0.58\nACT define_X\nWORDS\n")
L.check("ptolc_bridge docstring: on a bad word list it prints nothing and exits non-zero; with an EMPTY list it does so (so the C side keeps the word bag)", rc != 0 and out == "", f"rc={rc} out={out!r}: with no words it returns a sentence anyway")

'GAMMA -0.58\nACT define_X\nWORDS tree\n' -> (0, 'George trees.')


'GAMMA 0.30\nACT how\nWORDS tree plant grow\n' -> (0, 'Priest trees factory into factory.')


'GAMMA -0.58\nACT define_X\nWORDS\n' -> (0, 'Time is able.')


False

## The guaranteed fallback is real: with no usable HOME, ptol keeps the word bag

In [3]:
so, se, dt = ptol("-say", "what is a tree", real_home=False)
print("scratch HOME:", repr(so.strip()), f"{dt:.1f} s")
s_real, _, dt_real = ptol_say("what is a tree"); print("real HOME   :", repr(s_real), f"{dt_real:.1f} s")
L.check("with a scratch HOME the bridge cannot run and ptol keeps the word bag (fast), not a sentence", so.strip() != s_real and dt < dt_real, f"{dt:.1f} s vs {dt_real:.1f} s")

scratch HOME: 'polysyllables ise joost' 1.3 s


real HOME   : 'Time is able.' 12.6 s


True

## Ledger

In [4]:
_n = L.show()

LEDGER — 18 · the sentence path of the compiled monad
  [PASS ] ptol -say returns one non-empty line for each prompt
  [PASS ] the status line reports σ_self, Γ and the firing primes
  [PASS ] a valid word bag gives exit 0 and one sentence
          George trees.
  [FAIL ] ptolc_bridge docstring: on a bad word list it prints nothing and exits non-zero; with an EMPTY list it does so (so the C side keeps the word bag)
          rc=0 out='Time is able.': with no words it returns a sentence anyway
  [PASS ] with a scratch HOME the bridge cannot run and ptol keeps the word bag (fast), not a sentence
          1.3 s vs 12.6 s
  -> PASS 4 · FAIL 1 · PROSE 0
